In [ ]:
!pip install transformers torch accelerate bitsandbytes huggingface_hub

In [ ]:
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_name = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
print("Model loaded")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Model loaded


In [ ]:
from google.colab import files
uploaded = files.upload()  # train_set.csv and val_set.csv

import pandas as pd
train_df = pd.read_csv("train_set.csv")
val_df = pd.read_csv("val_set.csv")
print(f"Train: {len(train_df)}, Val: {len(val_df)}")

Saving train_set.csv to train_set (3).csv
Saving val_set.csv to val_set (2).csv
Train: 2768, Val: 395


**Zero-Shot Prompt, Parser, and Pathology columns**

In [ ]:
pathology_columns = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly',
    'Lung Lesion', 'Lung Opacity', 'Edema', 'Consolidation',
    'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion',
    'Pleural Other', 'Fracture', 'Support Devices'
]

def build_zero_shot_prompt(report_text):
    prompt = f"""You are a radiologist assistant. For each of the 14 findings below, provide a single label: 1 (Present), 0 (Absent), -1 (Uncertain), or NaN (Not Mentioned).

You must output exactly 14 lines, one for each finding. Even if a finding is not mentioned in the report, you must still output its line with the value NaN. Do not skip any line.

Output format (one finding per line):
No Finding=value
Enlarged Cardiomediastinum=value
Cardiomegaly=value
Lung Lesion=value
Lung Opacity=value
Edema=value
Consolidation=value
Pneumonia=value
Atelectasis=value
Pneumothorax=value
Pleural Effusion=value
Pleural Other=value
Fracture=value
Support Devices=value

Report:
{report_text}

Output:"""
    return prompt

def parse_model_output(output_text):
    valid_pathologies = [
        'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly',
        'Lung Lesion', 'Lung Opacity', 'Edema', 'Consolidation',
        'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion',
        'Pleural Other', 'Fracture', 'Support Devices'
    ]
    parsed = {}
    for line in output_text.strip().split("\n"):
        line = line.strip()
        if "=" in line:
            parts = line.split("=")
            if len(parts) == 2:
                name = parts[0].strip()
                value = parts[1].strip()
                if name in valid_pathologies and value in ["1", "0", "-1", "NaN", "nan"]:
                    parsed[name] = value
    result = []
    for p in valid_pathologies:
        result.append(parsed.get(p, "NaN"))
    return result

In [ ]:
# sanity check on one training report
sample = train_df.iloc[0]
report_text = sample['full_report']

prompt = build_zero_shot_prompt(report_text)
messages = [{"role": "user", "content": prompt}]
formatted_prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
output = model.generate(
    **inputs, max_new_tokens=200,
    do_sample=False, repetition_penalty=1.1
)

generated_ids = output[0][inputs["input_ids"].shape[1]:]
raw = tokenizer.decode(generated_ids, skip_special_tokens=True)

print("=== RAW OUTPUT ===")
print(raw)

predicted = parse_model_output(raw)
print("\n=== PARSED ===")
print(predicted)
print("Count:", len(predicted))

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


=== RAW OUTPUT ===
Here are the labels for each finding:

No Finding=0
Enlarged Cardiomediastinum=-1
Cardiomegaly=0
Lung Lesion=0
Lung Opacity=0
Edema=0
Consolidation=0
Pneumonia=0
Atelectasis=0
Pneumothorax=0
Pleural Effusion=0
Pleural Other=0
Fracture=0
Support Devices=0

=== PARSED ===
['0', '-1', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0']
Count: 14


In [ ]:
#zero-shot on validation set
def run_zero_shot_batch(df):
    results = []
    for i in range(len(df)):
        sample = df.iloc[i]
        report_text = sample['full_report']

        prompt = build_zero_shot_prompt(report_text)
        messages = [{"role": "user", "content": prompt}]
        formatted_prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
        output = model.generate(
            **inputs, max_new_tokens=200,
            do_sample=False, repetition_penalty=1.1
        )
        generated_ids = output[0][inputs["input_ids"].shape[1]:]
        raw = tokenizer.decode(generated_ids, skip_special_tokens=True)

        predicted = parse_model_output(raw)
        truth = sample[pathology_columns].values.tolist()

        results.append({'id': sample['id'], 'predicted': predicted, 'truth': truth})

        if (i + 1) % 50 == 0:
            print(f"Processed {i+1}/{len(df)}")

    return results

zero_shot_results = run_zero_shot_batch(val_df)
print(f"Completed zero-shot on {len(zero_shot_results)} validation reports")

Processed 50/395
Processed 100/395
Processed 150/395
Processed 200/395
Processed 250/395
Processed 300/395
Processed 350/395
Completed zero-shot on 395 validation reports


In [ ]:
def compute_metrics(results):
    metrics = {}
    for p_idx, pathology in enumerate(pathology_columns):
        tp = fp = fn = tn = 0
        for r in results:
            truth_val = r['truth'][p_idx]
            pred_val = r['predicted'][p_idx]
            if truth_val not in [0.0, 1.0]:
                continue
            try:
                pred_num = float(pred_val) if pred_val not in ["NaN", "nan"] else None
            except:
                pred_num = None
            if pred_num not in [0.0, 1.0]:
                continue
            truth_int = int(truth_val)
            pred_int = int(pred_num)
            if truth_int == 1 and pred_int == 1: tp += 1
            elif truth_int == 0 and pred_int == 1: fp += 1
            elif truth_int == 1 and pred_int == 0: fn += 1
            elif truth_int == 0 and pred_int == 0: tn += 1
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        metrics[pathology] = {'precision': precision, 'recall': recall, 'f1': f1,
                               'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn}
    return metrics

zero_shot_metrics = compute_metrics(zero_shot_results)

print(f"{'Pathology':30s} | {'Precision':>9} | {'Recall':>6} | {'F1':>6} | TP | FP | FN | TN")
print("-" * 90)
for pathology, m in zero_shot_metrics.items():
    print(f"{pathology:30s} | {m['precision']:>9.2f} | {m['recall']:>6.2f} | {m['f1']:>6.2f} | {m['tp']:>2} | {m['fp']:>2} | {m['fn']:>2} | {m['tn']:>2}")

macro_f1 = sum(m['f1'] for m in zero_shot_metrics.values()) / len(zero_shot_metrics)
print(f"\nMacro F1 (direct zero-shot): {macro_f1:.3f}")

Pathology                      | Precision | Recall |     F1 | TP | FP | FN | TN
------------------------------------------------------------------------------------------
No Finding                     |      0.00 |   0.00 |   0.00 |  0 |  0 | 39 |  0
Enlarged Cardiomediastinum     |      0.00 |   0.00 |   0.00 |  0 |  0 |  1 |  4
Cardiomegaly                   |      1.00 |   0.44 |   0.61 | 11 |  0 | 14 | 93
Lung Lesion                    |      1.00 |   1.00 |   1.00 | 10 |  0 |  0 |  7
Lung Opacity                   |      1.00 |   0.63 |   0.77 | 27 |  0 | 16 | 40
Edema                          |      0.75 |   1.00 |   0.86 |  3 |  1 |  0 |  6
Consolidation                  |      1.00 |   1.00 |   1.00 |  1 |  0 |  0 | 38
Pneumonia                      |      1.00 |   1.00 |   1.00 |  8 |  0 |  0 |  3
Atelectasis                    |      1.00 |   1.00 |   1.00 |  9 |  0 |  0 |  4
Pneumothorax                   |      1.00 |   0.33 |   0.50 |  1 |  0 |  2 | 107
Pleural Effusion 

In [ ]:
import pandas as pd

rows = []
for r in zero_shot_results:
    row = {'id': r['id']}
    for i, pathology in enumerate(pathology_columns):
        row[f'truth_{pathology}'] = r['truth'][i]
        row[f'pred_{pathology}'] = r['predicted'][i]
    rows.append(row)

pd.DataFrame(rows).to_csv("direct_zero_shot_results.csv", index=False)
print("Saved direct_zero_shot_results.csv")

from google.colab import files
files.download("direct_zero_shot_results.csv")

Saved direct_zero_shot_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**FEW-SHOT PROMPTING**

In [ ]:
def get_random_examples(train_df, n_examples, seed):
    return train_df.sample(n=n_examples, replace=False, random_state=seed)

def build_few_shot_prompt(report_text, examples):
    example_text = ""
    for i, (_, ex) in enumerate(examples.iterrows()):
        example_text += f"\nExample {i+1}:\nReport: {ex['full_report']}\n"
        for path in pathology_columns:
            val = ex[path]
            if pd.isna(val):
                example_text += f"{path}=NaN\n"
            elif val == 1.0:
                example_text += f"{path}=1\n"
            elif val == 0.0:
                example_text += f"{path}=0\n"
            elif val == -1.0:
                example_text += f"{path}=-1\n"

    prompt = f"""You are a radiologist assistant. For each of the 14 findings below, provide a single label: 1 (Present), 0 (Absent), -1 (Uncertain), or NaN (Not Mentioned).

You must output exactly 14 lines, one for each finding. Even if a finding is not mentioned in the report, you must still output its line with the value NaN. Do not skip any line.

Output format (one finding per line):
No Finding=value
Enlarged Cardiomediastinum=value
Cardiomegaly=value
Lung Lesion=value
Lung Opacity=value
Edema=value
Consolidation=value
Pneumonia=value
Atelectasis=value
Pneumothorax=value
Pleural Effusion=value
Pleural Other=value
Fracture=value
Support Devices=value
{example_text}
Now classify this report:
Report: {report_text}

Output:"""
    return prompt

def run_few_shot_batch(df, examples):
    results = []
    for i in range(len(df)):
        sample = df.iloc[i]
        report_text = sample['full_report']

        prompt = build_few_shot_prompt(report_text, examples)
        messages = [{"role": "user", "content": prompt}]
        formatted_prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
        output = model.generate(
            **inputs, max_new_tokens=200,
            do_sample=False, repetition_penalty=1.1
        )
        generated_ids = output[0][inputs["input_ids"].shape[1]:]
        raw = tokenizer.decode(generated_ids, skip_special_tokens=True)

        predicted = parse_model_output(raw)
        truth = sample[pathology_columns].values.tolist()

        results.append({'id': sample['id'], 'predicted': predicted, 'truth': truth})

        if (i + 1) % 50 == 0:
            print(f"Processed {i+1}/{len(df)}")

    return results

In [ ]:
import numpy as np

seeds = [42, 123, 456, 789, 1011]
macro_f1_scores = []
all_run_results = {}

for seed in seeds:
    print(f"\n=== Run with seed {seed} ===")
    examples = get_random_examples(train_df, n_examples=10, seed=seed)
    results = run_few_shot_batch(val_df, examples)
    metrics = compute_metrics(results)
    macro_f1 = sum(m['f1'] for m in metrics.values()) / len(metrics)
    macro_f1_scores.append(macro_f1)
    all_run_results[seed] = results
    print(f"Seed {seed}: macro F1 = {macro_f1:.3f}")

mean_f1 = np.mean(macro_f1_scores)
std_f1 = np.std(macro_f1_scores)
print(f"\n=== Aggregate ===")
print(f"All runs: {[f'{s:.3f}' for s in macro_f1_scores]}")
print(f"Mean ± Std: {mean_f1:.3f} ± {std_f1:.3f}")


=== Run with seed 42 ===
Processed 50/395
Processed 100/395
Processed 150/395
Processed 200/395
Processed 250/395
Processed 300/395
Processed 350/395
Seed 42: macro F1 = 0.633

=== Run with seed 123 ===
Processed 50/395
Processed 100/395
Processed 150/395
Processed 200/395
Processed 250/395
Processed 300/395
Processed 350/395
Seed 123: macro F1 = 0.738

=== Run with seed 456 ===
Processed 50/395
Processed 100/395
Processed 150/395
Processed 200/395
Processed 250/395
Processed 300/395
Processed 350/395
Seed 456: macro F1 = 0.655

=== Run with seed 789 ===
Processed 50/395
Processed 100/395
Processed 150/395
Processed 200/395
Processed 250/395
Processed 300/395
Processed 350/395
Seed 789: macro F1 = 0.715

=== Run with seed 1011 ===
Processed 50/395
Processed 100/395
Processed 150/395
Processed 200/395
Processed 250/395
Processed 300/395
Processed 350/395
Seed 1011: macro F1 = 0.719

=== Aggregate ===
All runs: ['0.633', '0.738', '0.655', '0.715', '0.719']
Mean ± Std: 0.692 ± 0.041


In [ ]:
import pandas as pd

for seed, results in all_run_results.items():
    rows = []
    for r in results:
        row = {'id': r['id']}
        for i, pathology in enumerate(pathology_columns):
            row[f'truth_{pathology}'] = r['truth'][i]
            row[f'pred_{pathology}'] = r['predicted'][i]
        rows.append(row)
    pd.DataFrame(rows).to_csv(f"direct_few_shot_seed{seed}_results.csv", index=False)
    print(f"Saved direct_few_shot_seed{seed}_results.csv")

from google.colab import files
for seed in all_run_results.keys():
    files.download(f"direct_few_shot_seed{seed}_results.csv")

Saved direct_few_shot_seed42_results.csv
Saved direct_few_shot_seed123_results.csv
Saved direct_few_shot_seed456_results.csv
Saved direct_few_shot_seed789_results.csv
Saved direct_few_shot_seed1011_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np

pathology_columns = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly',
    'Lung Lesion', 'Lung Opacity', 'Edema', 'Consolidation',
    'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion',
    'Pleural Other', 'Fracture', 'Support Devices'
]

def compute_metrics_from_csv(df):
    metrics = {}
    for p in pathology_columns:
        tp = fp = fn = tn = 0
        for _, row in df.iterrows():
            truth_val = row[f'truth_{p}']
            pred_val = row[f'pred_{p}']

            if pd.isna(truth_val) or truth_val not in [0.0, 1.0]:
                continue

            try:
                pred_num = float(pred_val) if str(pred_val).strip() not in ["NaN", "nan"] else None
            except:
                pred_num = None

            if pred_num not in [0.0, 1.0]:
                continue

            truth_int = int(truth_val)
            pred_int = int(pred_num)

            if truth_int == 1 and pred_int == 1: tp += 1
            elif truth_int == 0 and pred_int == 1: fp += 1
            elif truth_int == 1 and pred_int == 0: fn += 1
            elif truth_int == 0 and pred_int == 0: tn += 1

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        metrics[p] = {'precision': precision, 'recall': recall, 'f1': f1}

    return metrics

seeds = [42, 123, 456, 789, 1011]
macro_f1_scores = []

for seed in seeds:
    df = pd.read_csv(f"direct_few_shot_seed{seed}_results.csv")
    metrics = compute_metrics_from_csv(df)
    macro = np.mean([m['f1'] for m in metrics.values()])
    macro_f1_scores.append(macro)
    print(f"Seed {seed}: Macro F1 = {macro:.3f}")

# Zero-shot
zs_df = pd.read_csv("direct_zero_shot_results.csv")
zs_metrics = compute_metrics_from_csv(zs_df)
zs_macro = np.mean([m['f1'] for m in zs_metrics.values()])

print(f"\n=== Summary ===")
print(f"Zero-shot:  Macro F1 = {zs_macro:.3f}")
print(f"Few-shot:   Mean = {np.mean(macro_f1_scores):.3f}, Std = {np.std(macro_f1_scores):.3f}")
print(f"Individual runs: {[f'{s:.3f}' for s in macro_f1_scores]}")

Seed 42: Macro F1 = 0.633
Seed 123: Macro F1 = 0.738
Seed 456: Macro F1 = 0.655
Seed 789: Macro F1 = 0.715
Seed 1011: Macro F1 = 0.719

=== Summary ===
Zero-shot:  Macro F1 = 0.647
Few-shot:   Mean = 0.692, Std = 0.041
Individual runs: ['0.633', '0.738', '0.655', '0.715', '0.719']


In [ ]:
import pandas as pd, numpy as np

labels = ['No Finding','Enlarged Cardiomediastinum','Cardiomegaly','Lung Lesion','Lung Opacity',
          'Edema','Consolidation','Pneumonia','Atelectasis','Pneumothorax','Pleural Effusion',
          'Pleural Other','Fracture','Support Devices']
seeds = [42, 123, 456, 789, 1011]

def load(path):
    df = pd.read_csv(path)
    truth = df[[f'truth_{p}' for p in labels]].to_numpy(dtype=float)
    pred  = df[[f'pred_{p}' for p in labels]].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
    return df['id'].to_numpy(), truth, pred

ids_zs, truth_zs, pred_zs = load('direct_zero_shot_results.csv')
runs = []
for s in seeds:
    ids_s, t, p = load(f'direct_few_shot_seed{s}_results.csv')
    assert (ids_s == ids_zs).all(), f'row order differs for seed {s}'
    runs.append(p)

def f1_per_label(truth, pred):
    # -1 and NaN predictions are ignored
    t1, t0 = truth == 1, truth == 0
    p1, p0 = pred == 1, pred == 0
    tp = (t1 & p1).sum(0); fp = (t0 & p1).sum(0); fn = (t1 & p0).sum(0)
    prec = np.divide(tp, tp + fp, out=np.zeros_like(tp, float), where=(tp + fp) > 0)
    rec  = np.divide(tp, tp + fn, out=np.zeros_like(tp, float), where=(tp + fn) > 0)
    return np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(prec), where=(prec + rec) > 0)

zs = f1_per_label(truth_zs, pred_zs)
fs = np.mean([f1_per_label(truth_zs, p) for p in runs], axis=0)

print(f'{"label":30s} {"zero-shot":>10s} {"few-shot":>10s} {"change":>8s}')
for i, l in enumerate(labels):
    print(f'{l:30s} {zs[i]:10.3f} {fs[i]:10.3f} {fs[i]-zs[i]:+8.3f}')
print(f'{"MACRO F1":30s} {zs.mean():10.3f} {fs.mean():10.3f}')

# Draw the 395 reports at random (same report can be drawn twice), 1000 times,
# and see how often few-shot beats zero-shot when the val set is re-drawn.
rng = np.random.default_rng(0)
n = len(ids_zs)
diffs = []
for _ in range(1000):
    idx = rng.integers(0, n, n)
    z = f1_per_label(truth_zs[idx], pred_zs[idx]).mean()
    f = np.mean([f1_per_label(truth_zs[idx], p[idx]).mean() for p in runs])
    diffs.append(f - z)
diffs = np.array(diffs)
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f'\nfew-shot minus zero-shot, 95% interval: [{lo:+.3f}, {hi:+.3f}]')
print(f'share of re-draws where few-shot is ahead: {(diffs > 0).mean():.1%}')

label                           zero-shot   few-shot   change
No Finding                          0.000      0.998   +0.998
Enlarged Cardiomediastinum          0.000      0.106   +0.106
Cardiomegaly                        0.611      0.604   -0.007
Lung Lesion                         1.000      0.756   -0.244
Lung Opacity                        0.771      0.794   +0.022
Edema                               0.857      0.683   -0.174
Consolidation                       1.000      1.000   +0.000
Pneumonia                           1.000      0.804   -0.196
Atelectasis                         1.000      1.000   +0.000
Pneumothorax                        0.500      0.200   -0.300
Pleural Effusion                    0.800      0.580   -0.220
Pleural Other                       0.000      0.580   +0.580
Fracture                            0.923      0.689   -0.234
Support Devices                     0.593      0.895   +0.302
MACRO F1                            0.647      0.692

few-shot minus z